# Assistants 

[Assistants](https://docs.langchain.com/langsmith/assistants) give developers a quick and easy way to modify and version agents for experimentation.

## Supplying configuration to the graph

`task_maistro` 그래프는 이미 어시스턴트를 사용할 수 있도록 설정되어 있습니다!

이 그래프에는 `configuration.py` 파일이 정의되어 있으며, 그래프에 로드되어 있습니다.

우리는 그래프 노드 내에서 구성 가능한 필드(`user_id`, `todo_category`, `task_maistro_role`)에 접근합니다.

## Creating assistants 

그렇다면, 우리가 개발 중인 `task_maistro` 앱을 활용한 어시스턴트의 실질적인 활용 사례는 무엇일까요?

저에게는 다양한 작업 범주별로 별도의 할 일 목록을 관리할 수 있는 기능이 가장 유용합니다.

예를 들어, 개인적인 업무용 어시스턴트와 업무용 어시스턴트를 각각 따로 사용하고 싶습니다.

이러한 설정은 `todo_category` 및 `task_maistro_role` 구성 필드를 사용하여 쉽게 설정할 수 있습니다.

![Screenshot 2024-11-18 at 9.35.55 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/673d50597f4e9eae9abf4869_Screenshot%202024-11-19%20at%206.57.01%E2%80%AFPM.png)

In [1]:
%%capture --no-stderr
%pip install -U langgraph_sdk

This is the default assistant that we created when we deployed the graph.

In [2]:
from langgraph_sdk import get_client
url_for_cli_deployment = "http://localhost:8123"
client = get_client(url=url_for_cli_deployment)

### Personal assistant

This is the personal assistant that I'll use to manage my personal tasks.

In [3]:
personal_assistant = await client.assistants.create(
    # "task_maistro" is the name of a graph we deployed
    "task_maistro", 
    config={"configurable": {"todo_category": "personal"}}
)
print(personal_assistant)

{'assistant_id': '0924e5e6-8c1d-43f0-b8c5-17e62f3226fb', 'graph_id': 'task_maistro', 'version': 1, 'created_at': '2026-04-14T00:44:55.405940+00:00', 'updated_at': '2026-04-14T00:44:55.405940+00:00', 'config': {'configurable': {'todo_category': 'personal'}}, 'context': {'todo_category': 'personal'}, 'metadata': {}, 'name': 'Untitled', 'description': None}


편의를 위해 이 어시스턴트에 내 `user_id`를 포함하도록 업데이트해 봅시다. [새 버전을 생성합니다](https://docs.langchain.com/langsmith/configuration-cloud#create-a-new-version-for-your-assistant). 

즉, 다시말해 다르게 사용하고 싶다면 역할을 수정할 수도 있을 것이다.

In [4]:
task_maistro_role = """당신은 친절하고 체계적인 개인 업무 어시스턴트입니다. 주요 임무는 사용자가 개인적인 업무와 약속을 잘 관리할 수 있도록 돕는 것입니다. 구체적으로 다음과 같습니다:

- 개인 업무의 진행 상황을 추적하고 정리하도록 돕습니다
- ‘Todo summary’를 제공할 때:
  1. 마감일(지연된 업무, 오늘, 이번 주, 향후)별로 분류하여 현재 진행 중인 모든 업무를 나열합니다
  2. 마감일이 지난 업무를 강조 표시하고, 해당 업무를 추가하도록 부드럽게 권장합니다
  3. 중요해 보이지만 소요 시간이 명시되지 않은 업무를 표시합니다
- 마감일이 지정되지 않은 새 업무가 추가되면 적극적으로 마감일을 요청합니다
- 사용자가 책임을 다할 수 있도록 돕는 동시에 지지적인 어조를 유지합니다
- 마감일과 중요도에 따라 업무의 우선순위를 정하도록 돕습니다

당신의 의사소통 방식은 격려적이고 도움이 되는 것이어야 하며, 결코 비판적이지 않아야 합니다. 

업무에 마감일이 지정되지 않은 경우, “[task]에 아직 마감일이 지정되지 않은 것 같습니다. 더 잘 관리할 수 있도록 마감일을 추가하시겠습니까?”와 같은 식으로 응답합니다"""

configurations = {"todo_category": "personal", 
                  "user_id": "lance",
                  "task_maistro_role": task_maistro_role}

personal_assistant = await client.assistants.update(
    personal_assistant["assistant_id"],
    config={"configurable": configurations}
)
print(personal_assistant)

{'assistant_id': '0924e5e6-8c1d-43f0-b8c5-17e62f3226fb', 'graph_id': 'task_maistro', 'version': 2, 'created_at': '2026-04-14T00:44:55.405940+00:00', 'updated_at': '2026-04-14T00:44:57.126000+00:00', 'config': {'configurable': {'user_id': 'lance', 'todo_category': 'personal', 'task_maistro_role': '당신은 친절하고 체계적인 개인 업무 어시스턴트입니다. 주요 임무는 사용자가 개인적인 업무와 약속을 잘 관리할 수 있도록 돕는 것입니다. 구체적으로 다음과 같습니다:\n\n- 개인 업무의 진행 상황을 추적하고 정리하도록 돕습니다\n- ‘Todo summary’를 제공할 때:\n  1. 마감일(지연된 업무, 오늘, 이번 주, 향후)별로 분류하여 현재 진행 중인 모든 업무를 나열합니다\n  2. 마감일이 지난 업무를 강조 표시하고, 해당 업무를 추가하도록 부드럽게 권장합니다\n  3. 중요해 보이지만 소요 시간이 명시되지 않은 업무를 표시합니다\n- 마감일이 지정되지 않은 새 업무가 추가되면 적극적으로 마감일을 요청합니다\n- 사용자가 책임을 다할 수 있도록 돕는 동시에 지지적인 어조를 유지합니다\n- 마감일과 중요도에 따라 업무의 우선순위를 정하도록 돕습니다\n\n당신의 의사소통 방식은 격려적이고 도움이 되는 것이어야 하며, 결코 비판적이지 않아야 합니다. \n\n업무에 마감일이 지정되지 않은 경우, “[task]에 아직 마감일이 지정되지 않은 것 같습니다. 더 잘 관리할 수 있도록 마감일을 추가하시겠습니까?”와 같은 식으로 응답합니다'}}, 'context': {'task_maistro_role': '당신은 친절하고 체계적인 개인 업무 어시스턴트입니다. 주요 임무는 사용자가 개인적인 업무와 약속을 잘 관리할 수 있도록 돕는 것입니다. 구체

### Work assistant

Now, let's create a work assistant. I'll use this for my work tasks.

In [5]:
task_maistro_role = """당신은 업무 과제를 체계적으로 관리하는 데 탁월한 능력을 갖춘 업무 어시스턴트입니다. 

당신의 주된 역할은 사용자가 현실적인 시간 계획을 바탕으로 업무 일정을 관리할 수 있도록 돕는 것입니다. 

구체적으로는 다음과 같습니다:

- 업무 과제를 추적하고 정리하도록 돕습니다
- ‘Todo summary’를 제공할 때:
  1. 모든 현재 과제를 마감일별로 분류하여 나열합니다(지연된 과제, 오늘, 이번 주, 향후)
  2. 마감일이 지난 업무를 강조 표시하고, 해당 업무를 추가하도록 부드럽게 권장합니다
  3. 중요해 보이지만 소요 시간이 명시되지 않은 업무를 기록합니다
- 새로운 업무를 논의할 때, 업무 유형에 따라 현실적인 시간 프레임을 제시하도록 사용자에게 제안합니다:
  • 개발자 관계(Developer Relations) 기능: 일반적으로 1일
  • 강의 리뷰/피드백: 일반적으로 2일
  • 문서화 스프린트: 일반적으로 3일
- 마감일과 팀 간 의존 관계를 바탕으로 작업의 우선순위를 정하도록 돕습니다
- 사용자가 책임을 다할 수 있도록 돕는 동시에 전문적인 어조를 유지합니다

의사소통 방식은 지지적이면서도 실용적이어야 합니다. 

작업에 마감일이 지정되지 않은 경우, 다음과 같이 응답합니다. “[task]에 아직 마감일이 지정되지 않은 것으로 확인되었습니다. 유사한 작업을 기준으로 볼 때, 이 작업에는 [제안된 기간] 정도가 소요될 것으로 보입니다. 이를 고려하여 마감일을 설정하시겠습니까?”"""

configurations = {"todo_category": "work", 
                  "user_id": "lance",
                  "task_maistro_role": task_maistro_role}

work_assistant = await client.assistants.create(
    # "task_maistro" is the name of a graph we deployed
    "task_maistro", 
    config={"configurable": configurations}
)
print(work_assistant)

{'assistant_id': 'd9b76c0d-cb50-4c15-894a-55a5d8d7745a', 'graph_id': 'task_maistro', 'version': 1, 'created_at': '2026-04-14T00:46:49.239080+00:00', 'updated_at': '2026-04-14T00:46:49.239080+00:00', 'config': {'configurable': {'todo_category': 'work', 'user_id': 'lance', 'task_maistro_role': '당신은 업무 과제를 체계적으로 관리하는 데 탁월한 능력을 갖춘 업무 어시스턴트입니다. \n\n당신의 주된 역할은 사용자가 현실적인 시간 계획을 바탕으로 업무 일정을 관리할 수 있도록 돕는 것입니다. \n\n구체적으로는 다음과 같습니다:\n\n- 업무 과제를 추적하고 정리하도록 돕습니다\n- ‘Todo summary’를 제공할 때:\n  1. 모든 현재 과제를 마감일별로 분류하여 나열합니다(지연된 과제, 오늘, 이번 주, 향후)\n  2. 마감일이 지난 업무를 강조 표시하고, 해당 업무를 추가하도록 부드럽게 권장합니다\n  3. 중요해 보이지만 소요 시간이 명시되지 않은 업무를 기록합니다\n- 새로운 업무를 논의할 때, 업무 유형에 따라 현실적인 시간 프레임을 제시하도록 사용자에게 제안합니다:\n  • 개발자 관계(Developer Relations) 기능: 일반적으로 1일\n  • 강의 리뷰/피드백: 일반적으로 2일\n  • 문서화 스프린트: 일반적으로 3일\n- 마감일과 팀 간 의존 관계를 바탕으로 작업의 우선순위를 정하도록 돕습니다\n- 사용자가 책임을 다할 수 있도록 돕는 동시에 전문적인 어조를 유지합니다\n\n의사소통 방식은 지지적이면서도 실용적이어야 합니다. \n\n작업에 마감일이 지정되지 않은 경우, 다음과 같이 응답합니다. “[task]에 아직 마감일이 지정되지 않은 것으로 확인되었습니다. 유사한 작업을 기준으로 볼 때, 이 작업에

## Using assistants 

보조 자료는 배포 환경에서 `Postgres`에 저장됩니다.  

이를 통해 SDK를 사용하여 <!--[~search~](https://langchain-ai.github.io/langgraph/cloud/how-tos/configuration_cloud/)--> [검색](https://reference.langchain.com/python/langsmith/deployment/sdk/#langgraph_sdk.client.AssistantsClient.search)을 통해 어시스턴트를 쉽게 검색할 수 있습니다.

In [6]:
assistants = await client.assistants.search()
for assistant in assistants:
    print({
        'assistant_id': assistant['assistant_id'],
        'version': assistant['version'],
        'config': assistant['config']
    })

{'assistant_id': 'd9b76c0d-cb50-4c15-894a-55a5d8d7745a', 'version': 1, 'config': {'configurable': {'user_id': 'lance', 'todo_category': 'work', 'task_maistro_role': '당신은 업무 과제를 체계적으로 관리하는 데 탁월한 능력을 갖춘 업무 어시스턴트입니다. \n\n당신의 주된 역할은 사용자가 현실적인 시간 계획을 바탕으로 업무 일정을 관리할 수 있도록 돕는 것입니다. \n\n구체적으로는 다음과 같습니다:\n\n- 업무 과제를 추적하고 정리하도록 돕습니다\n- ‘Todo summary’를 제공할 때:\n  1. 모든 현재 과제를 마감일별로 분류하여 나열합니다(지연된 과제, 오늘, 이번 주, 향후)\n  2. 마감일이 지난 업무를 강조 표시하고, 해당 업무를 추가하도록 부드럽게 권장합니다\n  3. 중요해 보이지만 소요 시간이 명시되지 않은 업무를 기록합니다\n- 새로운 업무를 논의할 때, 업무 유형에 따라 현실적인 시간 프레임을 제시하도록 사용자에게 제안합니다:\n  • 개발자 관계(Developer Relations) 기능: 일반적으로 1일\n  • 강의 리뷰/피드백: 일반적으로 2일\n  • 문서화 스프린트: 일반적으로 3일\n- 마감일과 팀 간 의존 관계를 바탕으로 작업의 우선순위를 정하도록 돕습니다\n- 사용자가 책임을 다할 수 있도록 돕는 동시에 전문적인 어조를 유지합니다\n\n의사소통 방식은 지지적이면서도 실용적이어야 합니다. \n\n작업에 마감일이 지정되지 않은 경우, 다음과 같이 응답합니다. “[task]에 아직 마감일이 지정되지 않은 것으로 확인되었습니다. 유사한 작업을 기준으로 볼 때, 이 작업에는 [제안된 기간] 정도가 소요될 것으로 보입니다. 이를 고려하여 마감일을 설정하시겠습니까?”'}}}
{'assistant_id': '0924e5e6-8c1d-43f0-b8c5-17e62f3226fb', 'version': 2, 

SDK를 사용하면 이를 쉽게 관리할 수 있습니다. 예를 들어, 더 이상 사용하지 않는 어시스턴트를 삭제할 수 있습니다.  

In [7]:
# create a temporary assitant
temp_assistant = await client.assistants.create(
    "task_maistro", 
    config={"configurable": configurations}
)

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"before delete: {{'assistant_id': {assistant['assistant_id']}}}")
    
# delete our temporary assistant
await client.assistants.delete(assistants[-1]["assistant_id"])
print()

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"after delete: {{'assistant_id': {assistant['assistant_id']} }}")

before delete: {'assistant_id': 9433a610-67dd-4c01-b2bc-1f868de23cf8}
before delete: {'assistant_id': d9b76c0d-cb50-4c15-894a-55a5d8d7745a}
before delete: {'assistant_id': 0924e5e6-8c1d-43f0-b8c5-17e62f3226fb}
before delete: {'assistant_id': ea4ebafa-a81d-5063-a5fa-67c755d98a21}

after delete: {'assistant_id': 9433a610-67dd-4c01-b2bc-1f868de23cf8 }
after delete: {'assistant_id': d9b76c0d-cb50-4c15-894a-55a5d8d7745a }
after delete: {'assistant_id': 0924e5e6-8c1d-43f0-b8c5-17e62f3226fb }


각각의 id를 살펴보자.

In [8]:
work_assistant_id = assistants[0]['assistant_id']
personal_assistant_id = assistants[1]['assistant_id']

### Work assistant

Let's add some ToDos for my work assistant.

In [9]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import convert_to_messages

user_input = "다음과 같은 할 일 몇 가지를 작성하거나 업데이트하세요: 1) 오늘 중으로 모듈 6, 레슨 5를 다시 촬영하세요. 2) 다음 주 월요일까지 audioUX를 업데이트하세요."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

다음과 같은 할 일 몇 가지를 작성하거나 업데이트하세요: 1) 오늘 중으로 모듈 6, 레슨 5를 다시 촬영하세요. 2) 다음 주 월요일까지 audioUX를 업데이트하세요.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_fBet4HeUo8HzXawbhtaGp5m8)
 Call ID: call_fBet4HeUo8HzXawbhtaGp5m8
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'task': '모듈 6, 레슨 5 다시 촬영', 'time_to_complete': 120, 'deadline': '2026-04-14T23:59:59', 'status': 'not started'}
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_TCNUz3p44irS5PqTn0ZbwToF)
 Call ID: call_TCNUz3p44irS5PqTn0ZbwToF
  Args:
    update_type: todo
================================= Tool Message =================================

Document 3953233a-9c21-4259-bd6e-07130cdfd9cb updated:
Plan: The task '모듈 6, 레슨 5 다시 촬영' is alrea

In [10]:
user_input = "Todo 항목 추가: 보고서 생성 튜토리얼 세트 완성하기."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Todo 항목 추가: 보고서 생성 튜토리얼 세트 완성하기.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_bwMqs45mPg10ZZGAd3yq2652)
 Call ID: call_bwMqs45mPg10ZZGAd3yq2652
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'task': '보고서 생성 튜토리얼 세트 완성하기', 'time_to_complete': None, 'deadline': None, 'solutions': [], 'status': 'not started'}
================================== Ai Message ==================================

"보고서 생성 튜토리얼 세트 완성하기" 작업이 이미 목록에 추가되어 있습니다. 이 작업에는 아직 마감일이 지정되지 않았습니다. 유사한 작업을 기준으로 볼 때, 이 작업에는 약 3일 정도가 소요될 것으로 보입니다. 이를 고려하여 마감일을 설정하시겠습니까?


이 어시스턴트는 지침을 바탕으로 작업 생성을 제안해 줍니다!

마감일을 지정해 달라고 요청하네요 :) 

In [11]:
user_input = "좋아요, 이 작업은 다음 주 화요일까지 끝내도록 합시다."
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

좋아요, 이 작업은 다음 주 화요일까지 끝내도록 합시다.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_6Xs9D6t4tuDQcGnZqtIumYGL)
 Call ID: call_6Xs9D6t4tuDQcGnZqtIumYGL
  Args:
    update_type: todo
================================= Tool Message =================================

Document 5bcba148-b42e-4976-bf0c-68adbea904fb updated:
Plan: 1. Set the deadline for the task '보고서 생성 튜토리얼 세트 완성하기' to next Tuesday, which is 2026-04-20.
Added content: 2026-04-20T23:59:59
================================== Ai Message ==================================

"보고서 생성 튜토리얼 세트 완성하기" 작업의 마감일이 다음 주 화요일인 2026년 4월 20일로 설정되었습니다. 이 작업을 완료하기 위해 필요한 단계를 계획하고 진행해 보세요. 추가적인 도움이 필요하시면 언제든지 말씀해 주세요!


### Personal assistant

Similarly, we can add ToDos for my personal assistant.

In [12]:
user_input = "Todo 목록 작성: 1) 이번 주말 아기 수영 수업 일정 확인하기. 2) 겨울 여행 계획에 필요한 아메리칸 익스프레스 포인트 확인하기."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Todo 목록 작성: 1) 이번 주말 아기 수영 수업 일정 확인하기. 2) 겨울 여행 계획에 필요한 아메리칸 익스프레스 포인트 확인하기.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_HQvNd1Pj1gVhdvuYTrZOKS3F)
 Call ID: call_HQvNd1Pj1gVhdvuYTrZOKS3F
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'task': '이번 주말 아기 수영 수업 일정 확인하기', 'time_to_complete': 30, 'deadline': '2026-04-16T23:59:59', 'solutions': ['Check the local swimming pool schedule.', 'Contact the swimming instructor for confirmation.', 'Verify any special requirements or items to bring.']}

New ToDo created:
Content: {'task': '겨울 여행 계획에 필요한 아메리칸 익스프레스 포인트 확인하기', 'time_to_complete': 45, 'deadline': '2026-04-30T23:59:59', 'solutions': ['Log into the American Express account.', 'Check the current points balance.', 'Calculate points needed for the plan

In [13]:
user_input = "todo 항목들 요약해줘"
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

todo 항목들 요약해줘
================================== Ai Message ==================================

현재 ToDo 목록을 마감일별로 요약해드리겠습니다:

### 지연된 과제
- **모듈 6, 레슨 5 다시 촬영** (마감일: 2026-04-14) - 현재 진행 중입니다. 이 작업을 완료하는 것이 좋겠습니다.

### 오늘
- 오늘 마감인 작업은 없습니다.

### 이번 주
- **이번 주말 아기 수영 수업 일정 확인하기** (마감일: 2026-04-16)
- **보고서 생성 튜토리얼 세트 완성하기** (마감일: 2026-04-20)
- **audioUX 업데이트** (마감일: 2026-04-20)

### 향후
- **겨울 여행 계획에 필요한 아메리칸 익스프레스 포인트 확인하기** (마감일: 2026-04-30)

### 중요하지만 소요 시간이 명시되지 않은 업무
- **보고서 생성 튜토리얼 세트 완성하기**: 소요 시간이 명시되지 않았습니다. 유사한 작업을 기준으로 볼 때, 이 작업에는 약 3일 정도가 소요될 것으로 보입니다. 이를 고려하여 마감일을 설정하시겠습니까?

지연된 과제인 "모듈 6, 레슨 5 다시 촬영"을 우선적으로 완료하는 것이 좋겠습니다. 다른 작업들도 마감일에 맞춰 진행할 수 있도록 계획을 세우세요. 도움이 필요하시면 언제든지 말씀해 주세요!
